In [ ]:
# Importar bibliotecas
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report, confusion_matrix
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import SMOTE

# Opcional: evitar warnings innecesarios
warnings.filterwarnings('ignore')

# Configuración visual y de salida
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)


In [ ]:
# Cargar el archivo CSV (ahora usamos el dataset aumentado)
csv_path = '/Users/cristobal/Desktop/Capstone/neuroflex_dataset_augmented.csv'
df = pd.read_csv(csv_path)

# Limpiar la columna age para que quede como entero.
# Si el CSV la leyó como 67.0000, esto la convierte a 67.
# Nota: los datos reales no se modifican; esto solo asegura tipos correctos para el notebook.
df['age'] = pd.to_numeric(df['age'], errors='coerce')
# Mantener valores NA como 0 luego convertir a int
df['age'] = df['age'].fillna(0).round().astype(int)

# Mostrar contenido del CSV
print('Forma del dataset:', df.shape)
df.head()


In [ ]:
# Inspeccionar columnas y tipos de datos
print(df.info())
print('\nResumen estadístico:')
df.describe(include='all').T


In [ ]:
# Verificar valores faltantes
missing = df.isnull().sum()
print('Valores nulos por columna:')
print(missing[missing > 0])


In [ ]:
# Preparar variables para clasificación
# Definimos 3 niveles de rendimiento cognitivo según la precisión global del paciente.
# Revisamos la distribución real de Precision y ajustamos los umbrales para que las clases queden más equilibradas.
# Criterio nuevo y más balanceado:
# - Bajo: Precision < 35 -> ejecución con dificultades importantes
# - Medio: 35 <= Precision < 60 -> rendimiento moderado, con margen de mejora
# - Alto: Precision >= 60 -> buen rendimiento general
#
# Esto genera una clasificación multi-clase con una distribución más equilibrada y más fácil de aprender.

df['Precision'] = pd.to_numeric(df['Precision'], errors='coerce')
df['Precision'] = df['Precision'].clip(lower=0, upper=100)

df['rendimiento'] = pd.cut(
    df['Precision'],
    bins=[-1, 35, 60, 101],
    labels=['bajo', 'medio', 'alto'],
    right=False,
    include_lowest=True
)

df['target'] = df['rendimiento'].map({'bajo': 0, 'medio': 1, 'alto': 2})

# Verificamos la distribución de la etiqueta con los nuevos umbrales
print('Distribución de la clase objetivo con nuevos umbrales:')
print(df['target'].value_counts().sort_index())
print('Filas con target nulo:', df['target'].isna().sum())

# Eliminar filas con target inválido si llegaran a quedar
if df['target'].isna().any():
    df = df.dropna(subset=['target']).copy()

feature_cols = [
    'age',
    'TiempoRespuestaPararse',
    'TiempoActivoTarea',
    'CantAciertasTotales',
    'ObjetosInteractuadosCorrectamente',
    'TiempoRespuestaPregunta1',
    'TiempoRespuestaPregunta2',
    'TiempoRespuestaPregunta3',
    'TiempoCapturarNumero',
    'TiempoTutorial',
    'TipoEscena',
    'dominant_hand'
]

X = df[feature_cols]
y = df['target']

X.head()


In [ ]:
# Codificar variables categóricas
# La columna dominant_hand se codifica con OneHotEncoder
categorical_cols = ['dominant_hand']
numeric_cols = [col for col in feature_cols if col not in categorical_cols]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), numeric_cols),
        ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), categorical_cols)
    ]
)

preprocessor


In [ ]:
# Dividir datos en entrenamiento y prueba
# Usamos stratify para conservar la proporción de clases en train/test.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

print('Train size:', X_train.shape)
print('Test size:', X_test.shape)
print('\nDistribución de clases en train:')
print(y_train.value_counts().sort_index())
print('\nDistribución de clases en test:')
print(y_test.value_counts().sort_index())

# Preparación del preprocesador para la comparación de estrategias.
# Se usa solo sobre train para evitar fuga de información al test.
preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), numeric_cols),
        ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), categorical_cols)
    ]
)

X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc = preprocessor.transform(X_test)

# --- Estrategia 1: clasificación multiclase con SMOTE ---
smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train_proc, y_train)

print('\nDistribución de clases en train luego de SMOTE:')
print(pd.Series(y_train_bal).value_counts().sort_index())

# --- Estrategia 2: clasificación binaria 0 vs {1,2} ---
# Esto sirve como comparación para ver si simplificar el problema ayuda cuando la clase 2 es demasiado rara.
y_binary = (y >= 1).astype(int)

X_train_bin, X_test_bin, y_train_bin, y_test_bin = train_test_split(
    X,
    y_binary,
    test_size=0.25,
    random_state=42,
    stratify=y_binary
)

preprocessor_bin = ColumnTransformer(
    transformers=[
        ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), numeric_cols),
        ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), categorical_cols)
    ]
)

X_train_bin_proc = preprocessor_bin.fit_transform(X_train_bin)
X_test_bin_proc = preprocessor_bin.transform(X_test_bin)

smote_bin = SMOTE(random_state=42)
X_train_bin_bal, y_train_bin_bal = smote_bin.fit_resample(X_train_bin_proc, y_train_bin)

print('\nDistribución de clases binarias en train:')
print(pd.Series(y_train_bin_bal).value_counts().sort_index())


In [ ]:
# Entrenar el modelo multiclase con SMOTE
model_multiclass = RandomForestClassifier(
    n_estimators=500,
    random_state=42,
    class_weight='balanced'
)
model_multiclass.fit(X_train_bal, y_train_bal)
print('Modelo multiclase entrenado correctamente con SMOTE.')

# Entrenar el modelo binario 0 vs {1,2}
model_binary = RandomForestClassifier(
    n_estimators=500,
    random_state=42,
    class_weight='balanced'
)
model_binary.fit(X_train_bin_bal, y_train_bin_bal)
print('Modelo binario entrenado correctamente con SMOTE.')


In [ ]:
# Evaluar la estrategia multiclase con SMOTE
y_pred_multiclass = model_multiclass.predict(X_test_proc)

print('=== MODELO MULTICLASE CON SMOTE ===')
print('Accuracy:', round(accuracy_score(y_test, y_pred_multiclass), 4))
print('Balanced Accuracy:', round(balanced_accuracy_score(y_test, y_pred_multiclass), 4))
print('\nReporte de clasificación:')
print(classification_report(y_test, y_pred_multiclass, digits=4))
print('\nMatriz de confusión:')
print(confusion_matrix(y_test, y_pred_multiclass))

# Evaluar la estrategia binaria 0 vs {1,2}
y_pred_binary = model_binary.predict(X_test_bin_proc)

print('\n=== MODELO BINARIO 0 vs {1,2} ===')
print('Accuracy:', round(accuracy_score(y_test_bin, y_pred_binary), 4))
print('Balanced Accuracy:', round(balanced_accuracy_score(y_test_bin, y_pred_binary), 4))
print('\nReporte de clasificación:')
print(classification_report(y_test_bin, y_pred_binary, digits=4))
print('\nMatriz de confusión:')
print(confusion_matrix(y_test_bin, y_pred_binary))
